## Part 2: Prepare TLE Data
#### 1. Acquire the latest TLE Data from [space-track.org](https://www.space-track.org/)

In [8]:
# Reauthenticate with Space-Track.org to avoid having to run first cell again

# Space-Track.org API authentication 
from spacetrack import SpaceTrackClient
import pandas as pd

# read Space-Track.org credentials from external file (username in line 1, password in line 2)
with open('../DS_SpaceTrackCredentials.txt', 'r') as file:
    credentials = file.read().splitlines()

username = credentials[0]
password = credentials[1]

# create a SpaceTrackClient instance
st = SpaceTrackClient(username, password) 

try:
    # Try fetching a single SATCAT entry to verify authentication
    test_data = st.satcat(limit=1, format='json')
    print("Successfully authenticated with Space-Track.org!")
except Exception as e:
    print(f"Failed to authenticate with Space-Track.org: {e}")

print("Test data: " + test_data)

Successfully authenticated with Space-Track.org!
Test data: [{"INTLDES":"1957-001A","NORAD_CAT_ID":"1","OBJECT_TYPE":"ROCKET BODY","SATNAME":"SL-1 R\/B","COUNTRY":"CIS","LAUNCH":"1957-10-04","SITE":"TTMTR","DECAY":"1957-12-01","PERIOD":"96.19","INCLINATION":"65.10","APOGEE":"938","PERIGEE":"214","COMMENT":null,"COMMENTCODE":"4","RCSVALUE":"0","RCS_SIZE":"LARGE","FILE":"1","LAUNCH_YEAR":"1957","LAUNCH_NUM":"1","LAUNCH_PIECE":"A","CURRENT":"Y","OBJECT_NAME":"SL-1 R\/B","OBJECT_ID":"1957-001A","OBJECT_NUMBER":"1"}]


In [9]:
# Read the NORAD IDs from the file
norad_ids = pd.read_csv("../Data/DS_NoradIDs.csv")
print(f"Objects: {len(norad_ids)}")

Objects: 25700


In [10]:
# Split the IDs into chunks of 500 to avoid violating the API query restrictions
norad_ids_chunks = []
chunk_size = 500
for i in range(0, len(norad_ids), chunk_size):
    chunk = norad_ids[i:i + chunk_size]
    norad_ids_chunks.append(chunk)

# show number and size of chunks
print(f"Number of chunks: {len(norad_ids_chunks)}")
for i in range(len(norad_ids_chunks)):
    print(f"Chunk {i+1}: {len(norad_ids_chunks[i])}")

Number of chunks: 52
Chunk 1: 500
Chunk 2: 500
Chunk 3: 500
Chunk 4: 500
Chunk 5: 500
Chunk 6: 500
Chunk 7: 500
Chunk 8: 500
Chunk 9: 500
Chunk 10: 500
Chunk 11: 500
Chunk 12: 500
Chunk 13: 500
Chunk 14: 500
Chunk 15: 500
Chunk 16: 500
Chunk 17: 500
Chunk 18: 500
Chunk 19: 500
Chunk 20: 500
Chunk 21: 500
Chunk 22: 500
Chunk 23: 500
Chunk 24: 500
Chunk 25: 500
Chunk 26: 500
Chunk 27: 500
Chunk 28: 500
Chunk 29: 500
Chunk 30: 500
Chunk 31: 500
Chunk 32: 500
Chunk 33: 500
Chunk 34: 500
Chunk 35: 500
Chunk 36: 500
Chunk 37: 500
Chunk 38: 500
Chunk 39: 500
Chunk 40: 500
Chunk 41: 500
Chunk 42: 500
Chunk 43: 500
Chunk 44: 500
Chunk 45: 500
Chunk 46: 500
Chunk 47: 500
Chunk 48: 500
Chunk 49: 500
Chunk 50: 500
Chunk 51: 500
Chunk 52: 200


In [11]:
# Convert each chunk to a string to avoid violating the API query limit
chunk_strings=[]
for chunk in norad_ids_chunks:
    if not chunk.empty:
        chunk_string = ",".join(chunk["NORAD_CAT_ID"].astype(str))
        chunk_strings.append(chunk_string)


In [12]:
# Retreive TLE data for each chunk
import time # Import the time module to add delays between API requests
import json # Import the json module to parse the API response

tle_data_chunks = []
tle_data_received = 0

for i, chunk_string in enumerate(chunk_strings):
    tle_data = st.tle_latest(norad_cat_id=chunk_string, ordinal=1, format='json')
    tle_data_chunks.append(tle_data)

    # Parse the json response to count the elements
    tle_data_json = json.loads(tle_data)
    json_element_count = len(tle_data_json)

    print(f"Retrieved TLE data for chunk {i+1} with {len(chunk_string.split(','))} IDs requested and {json_element_count} elements returned")
    tle_data_received += json_element_count
    
    # Add a delay of 5 seconds between API requests to avoid rate limiting
    if i < len(chunk_strings) - 1:
        print("Waiting for 5 seconds before the next request...")
        time.sleep(5)

print(f"Total TLEs requested requested: {len(norad_ids)}")
print(f"Total TLE data elements received: {tle_data_received}")
print("Missing: " + str(len(norad_ids) - tle_data_received))

Retrieved TLE data for chunk 1 with 500 IDs requested and 474 elements returned
Waiting for 5 seconds before the next request...
Retrieved TLE data for chunk 2 with 500 IDs requested and 368 elements returned
Waiting for 5 seconds before the next request...
Retrieved TLE data for chunk 3 with 500 IDs requested and 409 elements returned
Waiting for 5 seconds before the next request...
Retrieved TLE data for chunk 4 with 500 IDs requested and 480 elements returned
Waiting for 5 seconds before the next request...
Retrieved TLE data for chunk 5 with 500 IDs requested and 490 elements returned
Waiting for 5 seconds before the next request...
Retrieved TLE data for chunk 6 with 500 IDs requested and 402 elements returned
Waiting for 5 seconds before the next request...
Retrieved TLE data for chunk 7 with 500 IDs requested and 427 elements returned
Waiting for 5 seconds before the next request...
Retrieved TLE data for chunk 8 with 500 IDs requested and 255 elements returned
Waiting for 5 sec

In [13]:
# Save the TLE data chunks to a dataframe
dataframes = [pd.DataFrame(json.loads(chunk)) for chunk in tle_data_chunks]
TLEs_df = pd.concat(dataframes, ignore_index=True)
TLEs_df

,ORDINAL,COMMENT,ORIGINATOR,NORAD_CAT_ID,OBJECT_NAME,OBJECT_TYPE,CLASSIFICATION_TYPE,INTLDES,EPOCH,EPOCH_MICROSECONDS,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,FILE,TLE_LINE0,TLE_LINE1,TLE_LINE2,OBJECT_ID,OBJECT_NUMBER,SEMIMAJOR_AXIS,PERIOD,APOGEE,PERIGEE,DECAYED
0,1,GENERATED VIA SPACETRACK.ORG API,18 SPCS,30,THOR ABLESTAR R/B,ROCKET BODY,U,60003A,1961-08-16 14:24:53,774496,16.29358349,0,51.25,356.4892,331.9473,28.0526,0,999,7590,0,0.02125614,0,33907,0 THOR ABLESTAR R/B,1 00030U 60003 A 61228.60062239 .02125614 +0...,2 00030 051.2500 356.4892 0000000 331.9473 028...,1960-003A,30,6572.406,88.378,194.271,194.271,1
1,1,GENERATED VIA SPACETRACK.ORG API,18 SPCS,33,THOR ABLESTAR DEB,DEBRIS,U,60003C,1960-10-06 07:59:06,496512,15.38343354,0.0242757,51.2879,136.0668,196.78,163.61,0,999,2700,0,0.00001932,0,33906,0 THOR ABLESTAR DEB,1 00033U 60003 C 60280.33271408 .00001932 +0...,2 00033 051.2879 136.0668 0242757 196.7800 163...,1960-003C,33,6829.149,93.607,616.797,285.232,1
2,1,GENERATED VIA SPACETRACK.ORG API,18 SPCS,35,SL-3 R/B,ROCKET BODY,U,60005B,1960-07-16 04:00:11,150208,16.27523673,0,64.8899,44.6976,36.3819,323.618,0,999,990,0,0.03271057,0,33906,0 SL-3 R/B,1 00035U 60005 B 60198.16679572 .03271057 +0...,2 00035 064.8899 044.6976 0000000 036.3819 323...,1960-005B,35,6577.344,88.478,199.209,199.209,1
3,1,GENERATED VIA SPACETRACK.ORG API,18 SPCS,37,SPUTNIK 4 DEB,DEBRIS,U,60005D,1961-03-15 15:48:42,802271,15.49800338,0.0198403,64.8899,343.2411,333.4201,27.084,0,999,4690,0,0.00039007,0,33907,0 SPUTNIK 4 DEB,1 00037U 60005 D 61074.65882873 .00039007 +0...,2 00037 064.8899 343.2411 0198403 333.4201 027...,1960-005D,37,6795.451,92.915,552.140,282.492,1
4,1,GENERATED VIA SPACETRACK.ORG API,18 SPCS,38,SPUTNIK 4 DEB,DEBRIS,U,60005E,1960-08-19 16:34:33,722976,15.88909013,0.0034213,64.8899,302.1218,49.4823,310.6691,0,999,1500,0,0.00610805,0,33906,0 SPUTNIK 4 DEB,1 00038U 60005 E 60232.69066809 .00610805 +0...,2 00038 064.8899 302.1218 0034213 049.4823 310...,1960-005E,38,6683.482,90.628,328.213,282.481,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24599,1,GENERATED VIA SPACETRACK.ORG API,18 SPCS,61384,CZ-6A DEB,DEBRIS,U,24140WZ,2024-11-06 15:41:58,700832,16.34116299,0.0015079,88.9497,338.4494,47.4482,312.8094,0,999,958,0.0034896,0.23331396,-0.0000011521,4547960,0 CZ-6A DEB,1 61384U 24140WZ 24311.65415163 .23331396 -1...,2 61384 88.9497 338.4494 0015079 47.4482 312...,2024-140WZ,61384,6559.638,88.121,191.394,171.612,1
24600,1,GENERATED VIA SPACETRACK.ORG API,18 SPCS,61407,CZ-6A DEB,DEBRIS,U,24140XY,2024-12-11 23:05:15,774432,15.90490446,0.0018751,89.207,338.9347,209.0449,150.9793,0,999,1395,0.031616,0.07121792,0.091495,4579024,0 CZ-6A DEB,1 61407U 24140XY 24346.96198813 .07121792 9...,2 61407 89.2070 338.9347 0018751 209.0449 150...,2024-140XY,61407,6679.046,90.538,313.435,288.388,1
24601,1,GENERATED VIA SPACETRACK.ORG API,18 SPCS,61438,CZ-6A DEB,DEBRIS,U,24140ZF,2024-11-13 06:57:45,539424,16.26800607,0.0030332,90.0847,353.4507,44.4626,315.9104,0,999,1498,0.0048661,0.17891631,-0.00000054261,4552726,0 CZ-6A DEB,1 61438U 24140ZF 24318.29011041 .17891631 -5...,2 61438 90.0847 353.4507 0030332 44.4626 315...,2024-140ZF,61438,6579.289,88.517,221.110,181.198,1
24602,1,GENERATED VIA SPACETRACK.ORG API,18 SPCS,61445,CZ-2D R/B,ROCKET BODY,U,24177B,2024-11-10 05:25:56,203680,16.37905997,0.0007529,41.3775,239.1072,234.7996,251.2891,0,999,699,0.00067146,0.07484936,0.0000079825,4549416,0 CZ-2D R/B,1 61445U 24177B 24315.22634495 .07484936 7...,2 61445 41.3775 239.1072 0007529 234.7996 251...,2024-177B,61445,6549.516,87.917,176.312,166.450,1


In [14]:
#show columns
print(TLEs_df.columns)

Index(['ORDINAL', 'COMMENT', 'ORIGINATOR', 'NORAD_CAT_ID', 'OBJECT_NAME',
       'OBJECT_TYPE', 'CLASSIFICATION_TYPE', 'INTLDES', 'EPOCH',
       'EPOCH_MICROSECONDS', 'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION',
       'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'MEAN_ANOMALY', 'EPHEMERIS_TYPE',
       'ELEMENT_SET_NO', 'REV_AT_EPOCH', 'BSTAR', 'MEAN_MOTION_DOT',
       'MEAN_MOTION_DDOT', 'FILE', 'TLE_LINE0', 'TLE_LINE1', 'TLE_LINE2',
       'OBJECT_ID', 'OBJECT_NUMBER', 'SEMIMAJOR_AXIS', 'PERIOD', 'APOGEE',
       'PERIGEE', 'DECAYED'],
      dtype='object')


In [15]:
# show all columns
pd.set_option('display.max_columns', None)
TLEs_df

,ORDINAL,COMMENT,ORIGINATOR,NORAD_CAT_ID,OBJECT_NAME,OBJECT_TYPE,CLASSIFICATION_TYPE,INTLDES,EPOCH,EPOCH_MICROSECONDS,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,FILE,TLE_LINE0,TLE_LINE1,TLE_LINE2,OBJECT_ID,OBJECT_NUMBER,SEMIMAJOR_AXIS,PERIOD,APOGEE,PERIGEE,DECAYED
0,1,GENERATED VIA SPACETRACK.ORG API,18 SPCS,30,THOR ABLESTAR R/B,ROCKET BODY,U,60003A,1961-08-16 14:24:53,774496,16.29358349,0,51.25,356.4892,331.9473,28.0526,0,999,7590,0,0.02125614,0,33907,0 THOR ABLESTAR R/B,1 00030U 60003 A 61228.60062239 .02125614 +0...,2 00030 051.2500 356.4892 0000000 331.9473 028...,1960-003A,30,6572.406,88.378,194.271,194.271,1
1,1,GENERATED VIA SPACETRACK.ORG API,18 SPCS,33,THOR ABLESTAR DEB,DEBRIS,U,60003C,1960-10-06 07:59:06,496512,15.38343354,0.0242757,51.2879,136.0668,196.78,163.61,0,999,2700,0,0.00001932,0,33906,0 THOR ABLESTAR DEB,1 00033U 60003 C 60280.33271408 .00001932 +0...,2 00033 051.2879 136.0668 0242757 196.7800 163...,1960-003C,33,6829.149,93.607,616.797,285.232,1
2,1,GENERATED VIA SPACETRACK.ORG API,18 SPCS,35,SL-3 R/B,ROCKET BODY,U,60005B,1960-07-16 04:00:11,150208,16.27523673,0,64.8899,44.6976,36.3819,323.618,0,999,990,0,0.03271057,0,33906,0 SL-3 R/B,1 00035U 60005 B 60198.16679572 .03271057 +0...,2 00035 064.8899 044.6976 0000000 036.3819 323...,1960-005B,35,6577.344,88.478,199.209,199.209,1
3,1,GENERATED VIA SPACETRACK.ORG API,18 SPCS,37,SPUTNIK 4 DEB,DEBRIS,U,60005D,1961-03-15 15:48:42,802271,15.49800338,0.0198403,64.8899,343.2411,333.4201,27.084,0,999,4690,0,0.00039007,0,33907,0 SPUTNIK 4 DEB,1 00037U 60005 D 61074.65882873 .00039007 +0...,2 00037 064.8899 343.2411 0198403 333.4201 027...,1960-005D,37,6795.451,92.915,552.140,282.492,1
4,1,GENERATED VIA SPACETRACK.ORG API,18 SPCS,38,SPUTNIK 4 DEB,DEBRIS,U,60005E,1960-08-19 16:34:33,722976,15.88909013,0.0034213,64.8899,302.1218,49.4823,310.6691,0,999,1500,0,0.00610805,0,33906,0 SPUTNIK 4 DEB,1 00038U 60005 E 60232.69066809 .00610805 +0...,2 00038 064.8899 302.1218 0034213 049.4823 310...,1960-005E,38,6683.482,90.628,328.213,282.481,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24599,1,GENERATED VIA SPACETRACK.ORG API,18 SPCS,61384,CZ-6A DEB,DEBRIS,U,24140WZ,2024-11-06 15:41:58,700832,16.34116299,0.0015079,88.9497,338.4494,47.4482,312.8094,0,999,958,0.0034896,0.23331396,-0.0000011521,4547960,0 CZ-6A DEB,1 61384U 24140WZ 24311.65415163 .23331396 -1...,2 61384 88.9497 338.4494 0015079 47.4482 312...,2024-140WZ,61384,6559.638,88.121,191.394,171.612,1
24600,1,GENERATED VIA SPACETRACK.ORG API,18 SPCS,61407,CZ-6A DEB,DEBRIS,U,24140XY,2024-12-11 23:05:15,774432,15.90490446,0.0018751,89.207,338.9347,209.0449,150.9793,0,999,1395,0.031616,0.07121792,0.091495,4579024,0 CZ-6A DEB,1 61407U 24140XY 24346.96198813 .07121792 9...,2 61407 89.2070 338.9347 0018751 209.0449 150...,2024-140XY,61407,6679.046,90.538,313.435,288.388,1
24601,1,GENERATED VIA SPACETRACK.ORG API,18 SPCS,61438,CZ-6A DEB,DEBRIS,U,24140ZF,2024-11-13 06:57:45,539424,16.26800607,0.0030332,90.0847,353.4507,44.4626,315.9104,0,999,1498,0.0048661,0.17891631,-0.00000054261,4552726,0 CZ-6A DEB,1 61438U 24140ZF 24318.29011041 .17891631 -5...,2 61438 90.0847 353.4507 0030332 44.4626 315...,2024-140ZF,61438,6579.289,88.517,221.110,181.198,1
24602,1,GENERATED VIA SPACETRACK.ORG API,18 SPCS,61445,CZ-2D R/B,ROCKET BODY,U,24177B,2024-11-10 05:25:56,203680,16.37905997,0.0007529,41.3775,239.1072,234.7996,251.2891,0,999,699,0.00067146,0.07484936,0.0000079825,4549416,0 CZ-2D R/B,1 61445U 24177B 24315.22634495 .07484936 7...,2 61445 41.3775 239.1072 0007529 234.7996 251...,2024-177B,61445,6549.516,87.917,176.312,166.450,1


In [16]:
# drop not needed columns
TLEs_df.drop(columns=['ORDINAL', 'COMMENT', 'ORIGINATOR', 'CLASSIFICATION_TYPE', 'EPHEMERIS_TYPE', 'ELEMENT_SET_NO', 'FILE', 'DECAYED'], inplace=True)
TLEs_df

,NORAD_CAT_ID,OBJECT_NAME,OBJECT_TYPE,INTLDES,EPOCH,EPOCH_MICROSECONDS,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,TLE_LINE0,TLE_LINE1,TLE_LINE2,OBJECT_ID,OBJECT_NUMBER,SEMIMAJOR_AXIS,PERIOD,APOGEE,PERIGEE
0,30,THOR ABLESTAR R/B,ROCKET BODY,60003A,1961-08-16 14:24:53,774496,16.29358349,0,51.25,356.4892,331.9473,28.0526,7590,0,0.02125614,0,0 THOR ABLESTAR R/B,1 00030U 60003 A 61228.60062239 .02125614 +0...,2 00030 051.2500 356.4892 0000000 331.9473 028...,1960-003A,30,6572.406,88.378,194.271,194.271
1,33,THOR ABLESTAR DEB,DEBRIS,60003C,1960-10-06 07:59:06,496512,15.38343354,0.0242757,51.2879,136.0668,196.78,163.61,2700,0,0.00001932,0,0 THOR ABLESTAR DEB,1 00033U 60003 C 60280.33271408 .00001932 +0...,2 00033 051.2879 136.0668 0242757 196.7800 163...,1960-003C,33,6829.149,93.607,616.797,285.232
2,35,SL-3 R/B,ROCKET BODY,60005B,1960-07-16 04:00:11,150208,16.27523673,0,64.8899,44.6976,36.3819,323.618,990,0,0.03271057,0,0 SL-3 R/B,1 00035U 60005 B 60198.16679572 .03271057 +0...,2 00035 064.8899 044.6976 0000000 036.3819 323...,1960-005B,35,6577.344,88.478,199.209,199.209
3,37,SPUTNIK 4 DEB,DEBRIS,60005D,1961-03-15 15:48:42,802271,15.49800338,0.0198403,64.8899,343.2411,333.4201,27.084,4690,0,0.00039007,0,0 SPUTNIK 4 DEB,1 00037U 60005 D 61074.65882873 .00039007 +0...,2 00037 064.8899 343.2411 0198403 333.4201 027...,1960-005D,37,6795.451,92.915,552.140,282.492
4,38,SPUTNIK 4 DEB,DEBRIS,60005E,1960-08-19 16:34:33,722976,15.88909013,0.0034213,64.8899,302.1218,49.4823,310.6691,1500,0,0.00610805,0,0 SPUTNIK 4 DEB,1 00038U 60005 E 60232.69066809 .00610805 +0...,2 00038 064.8899 302.1218 0034213 049.4823 310...,1960-005E,38,6683.482,90.628,328.213,282.481
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24599,61384,CZ-6A DEB,DEBRIS,24140WZ,2024-11-06 15:41:58,700832,16.34116299,0.0015079,88.9497,338.4494,47.4482,312.8094,958,0.0034896,0.23331396,-0.0000011521,0 CZ-6A DEB,1 61384U 24140WZ 24311.65415163 .23331396 -1...,2 61384 88.9497 338.4494 0015079 47.4482 312...,2024-140WZ,61384,6559.638,88.121,191.394,171.612
24600,61407,CZ-6A DEB,DEBRIS,24140XY,2024-12-11 23:05:15,774432,15.90490446,0.0018751,89.207,338.9347,209.0449,150.9793,1395,0.031616,0.07121792,0.091495,0 CZ-6A DEB,1 61407U 24140XY 24346.96198813 .07121792 9...,2 61407 89.2070 338.9347 0018751 209.0449 150...,2024-140XY,61407,6679.046,90.538,313.435,288.388
24601,61438,CZ-6A DEB,DEBRIS,24140ZF,2024-11-13 06:57:45,539424,16.26800607,0.0030332,90.0847,353.4507,44.4626,315.9104,1498,0.0048661,0.17891631,-0.00000054261,0 CZ-6A DEB,1 61438U 24140ZF 24318.29011041 .17891631 -5...,2 61438 90.0847 353.4507 0030332 44.4626 315...,2024-140ZF,61438,6579.289,88.517,221.110,181.198
24602,61445,CZ-2D R/B,ROCKET BODY,24177B,2024-11-10 05:25:56,203680,16.37905997,0.0007529,41.3775,239.1072,234.7996,251.2891,699,0.00067146,0.07484936,0.0000079825,0 CZ-2D R/B,1 61445U 24177B 24315.22634495 .07484936 7...,2 61445 41.3775 239.1072 0007529 234.7996 251...,2024-177B,61445,6549.516,87.917,176.312,166.450


In [17]:
# Save TLEs to csv
TLEs_df.to_csv('../Data/DS_TLEs.csv', index=False)